In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip "/content/drive/MyDrive/Online Proctoring.zip" -d "/content/drive/MyDrive/Online Proctoring"

Streaming output truncated to the last 5000 lines.
  inflating: /content/drive/MyDrive/Online Proctoring/Online Proctoring/train/labels/final-2_mp4-8669_jpg.rf.f799cb441baffeca089988978cc6cfbe.txt  
  inflating: /content/drive/MyDrive/Online Proctoring/Online Proctoring/train/labels/final-2_mp4-8670_jpg.rf.406f5f65e5b3aefd6a56f7396ab52a6e.txt  
  inflating: /content/drive/MyDrive/Online Proctoring/Online Proctoring/train/labels/final-2_mp4-8671_jpg.rf.8ad44d09a4622afe1ce93135b3d66be9.txt  
  inflating: /content/drive/MyDrive/Online Proctoring/Online Proctoring/train/labels/final-2_mp4-8672_jpg.rf.15fd9c56f48efaadd9fa6012e4f80d61.txt  
  inflating: /content/drive/MyDrive/Online Proctoring/Online Proctoring/train/labels/final-2_mp4-8673_jpg.rf.e7a1b2103d47a9b273fbe27595a2f25c.txt  
  inflating: /content/drive/MyDrive/Online Proctoring/Online Proctoring/train/labels/final-2_mp4-8674_jpg.rf.5c770f54e93716787f560606e216e7d0.txt  
  inflating: /content/drive/MyDrive/Online Proctoring/Online 

In [5]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00


In [3]:
import yaml
import os

DATASET_ROOT = "/content/drive/MyDrive/Online Proctoring/Online Proctoring"
DATASET_YAML = os.path.join(DATASET_ROOT, "data.yaml")

with open(DATASET_YAML, "r") as f:
    cfg = yaml.safe_load(f)

cfg["train"] = os.path.join(DATASET_ROOT, "train", "images")
cfg["val"]   = os.path.join(DATASET_ROOT, "valid", "images")
cfg["test"]  = os.path.join(DATASET_ROOT, "test",  "images")

with open(DATASET_YAML, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("data.yaml updated:")
with open(DATASET_YAML) as f:
    print(f.read())

data.yaml updated:
names:
- book
- cell phone
- headphone
- laptop
- person
- tv
nc: 6
roboflow:
  license: CC BY 4.0
  project: online-proctoring-system-x27ou-nl8r1-intbk
  url: https://universe.roboflow.com/ahmed-mohamed-sifjd/online-proctoring-system-x27ou-nl8r1-intbk/dataset/1
  version: 1
  workspace: ahmed-mohamed-sifjd
test: /content/drive/MyDrive/Online Proctoring/Online Proctoring/test/images
train: /content/drive/MyDrive/Online Proctoring/Online Proctoring/train/images
val: /content/drive/MyDrive/Online Proctoring/Online Proctoring/valid/images



In [4]:
import torch, glob

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4
  train: 0 images | 0 labels
  valid: 0 images | 0 labels
  test: 0 images | 0 labels


In [6]:
from ultralytics import YOLO

MODEL_SIZE  = "yolo26s"
EPOCHS      = 50
IMG_SIZE    = 640
BATCH_SIZE  = 16
LR0         = 0.01
PROJECT     = "proctoring"
RUN_NAME    = "yolo26s_run1"

model = YOLO(f"{MODEL_SIZE}.pt")

results = model.train(
    data      = DATASET_YAML,
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    lr0       = LR0,
    project   = PROJECT,
    name      = RUN_NAME,
    patience  = 15,
    save      = True,
    plots     = True,
    cache     = True,
    device    = 0,
    workers   = 2,
)
print("Best weights:", results.save_dir)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Online Proctoring/Online Proctoring/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

In [11]:
BEST_WEIGHTS = "/content/runs/detect/proctoring/yolo26s_run1/weights/best.pt"
model_best   = YOLO(BEST_WEIGHTS)

val_metrics  = model_best.val(data=DATASET_YAML, split="val")
print(f"mAP50: {val_metrics.box.map50:.4f}  |  mAP50-95: {val_metrics.box.map:.4f}")

test_metrics = model_best.val(data=DATASET_YAML, split="test")
print(f"mAP50: {test_metrics.box.map50:.4f}  |  mAP50-95: {test_metrics.box.map:.4f}")

Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,467,502 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 7.6±3.4 MB/s, size: 26.0 KB)
val: Scanning /content/drive/MyDrive/Online Proctoring/Online Proctoring/valid/labels.cache... 1822 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1822/1822 509.5Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 63, len(boxes) = 4046. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 114/114 2.5it/s 45.5s
                   all       1822       4046      0.896      0.918      0.928      0.622
                  book        204        204      0.908      0.961      0.951      0.